## Imports

In [1]:
from pathlib import Path
import sys
import os

In [2]:
import numpy as np

import pycuda.autoinit
import pycuda.driver as cuda
from pycuda.compiler import SourceModule

In [3]:
project_working_dir = str(Path(sys.path[0]).parent)
sys.path += [project_working_dir]
os.chdir(project_working_dir)

In [4]:
from src.utils.file_io import read_file_str, show_formatted_cpp

In [5]:
!cl

usage: cl [ option... ] filename... [ /link linkoption... ]


Microsoft (R) C/C++ Optimizing Compiler Version 19.43.34810 for x64
Copyright (C) Microsoft Corporation.  All rights reserved.



## Create some dummy data

In [6]:
vertex_data = np.array(
    [
        [1, 0, 1, 0, 0, 0, 0, 1],
        [0, 2, 0, 0, 0, 0, 0, 1],
        [-1, 0, 1, 0, 0, 0, 0, 1],
        [-1, 0, -1, 0, 0, 0, 0, 1],
        [1, 0, -1, 0, 0, 0, 0, 1],
    ],
    dtype=np.float32,
)

In [7]:
triangle_data = np.array(
    [
        [0, 1, 2],
        [3, 1, 4],
    ],
    dtype=np.uint32,
)

## Cuda Parameters

In [8]:
BLOCK_SIZE = 1024
NR_TRIANGLE_BLOCKS = (len(triangle_data) + BLOCK_SIZE - 1) // BLOCK_SIZE
NR_VERTEX_BLOCKS = (len(triangle_data) + BLOCK_SIZE - 1) // BLOCK_SIZE

## Compile cuda kernel

In [9]:
cuda_code = read_file_str("./profiling/kernels/update_normals.cu")

In [10]:
show_formatted_cpp(cuda_code)

In [11]:
mod = SourceModule(cuda_code)

## Set-up memory for running kernel

In [12]:
zero_out_normals = mod.get_function("zero_out_normals")

In [13]:
sum_normals = mod.get_function("sum_normals_over_triangles")

In [14]:
normalise_normals = mod.get_function("normalize_normals")

In [15]:
nr_vertices = np.uint32(len(vertex_data))

In [16]:
nr_traingles = np.uint32(len(triangle_data))

### Allocate memory to gpu

In [17]:
assert vertex_data.flatten().flags["C_CONTIGUOUS"]
assert triangle_data.flatten().flags["C_CONTIGUOUS"]

In [18]:
vertex_data_gpu = cuda.mem_alloc(vertex_data.nbytes)

In [19]:
cuda.memcpy_htod(vertex_data_gpu, vertex_data.flatten())

In [20]:
triangle_data_gpu = cuda.mem_alloc(triangle_data.nbytes)

In [21]:
cuda.memcpy_htod(triangle_data_gpu, triangle_data.flatten())

## Profile function

In [22]:
def apply_normal_update():
    zero_out_normals(
        vertex_data_gpu,
        nr_vertices,
        block=(BLOCK_SIZE, 1, 1),
        grid=(NR_VERTEX_BLOCKS, 1, 1),
    )
    sum_normals(
        vertex_data_gpu,
        triangle_data_gpu,
        nr_traingles,
        block=(BLOCK_SIZE, 1, 1),
        grid=(NR_TRIANGLE_BLOCKS, 1, 1),
    )
    normalise_normals(
        vertex_data_gpu,
        nr_vertices,
        block=(BLOCK_SIZE, 1, 1),
        grid=(NR_VERTEX_BLOCKS, 1, 1),
    )

In [23]:
%timeit apply_normal_update()

42.6 µs ± 2.92 µs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


## Check output is as expected

In [24]:
cuda.memcpy_dtoh(vertex_data, vertex_data_gpu)

In [25]:
vertex_data[:, 5:]

array([[ 0.       ,  0.4472136,  0.8944272],
       [ 0.       ,  1.       ,  0.       ],
       [ 0.       ,  0.4472136,  0.8944272],
       [ 0.       ,  0.4472136, -0.8944272],
       [ 0.       ,  0.4472136, -0.8944272]], dtype=float32)

In [26]:
assert np.all(vertex_data[1, 5:] == [0, 1, 0])